In [6]:
import torch

from ipynb.fs.defs.data import reinforcement_data
from ipynb.fs.defs.features import build_features
from ipynb.fs.defs.q_network import QNetwork
from ipynb.fs.defs.save_load import load_model

In [7]:
model = load_model()

ACTION_MAP = {
    0: "HOLD",
    1: "BUY 50%",
    2: "BUY 100%",
    3: "SELL 50%",
    4: "SELL 100%"
}

Model loaded from models/dqn_trader.pt


In [8]:
def predict_signal(
    ticker,
    position_fraction=0.0,
    unrealized_pnl=0.0
):

    prices = reinforcement_data(
        ticker,
        "2023-01-01",
        "2025-12-31"
    ).squeeze()

    features = build_features(prices)

    latest = features.iloc[-1]

    state = [
        latest["momentum"],
        latest["volatility"],
        latest["ma_signal"],
        latest["rsi"],
        latest["return_20"],
        latest["return_50"],
        latest["ma50_signal"],
        latest["ma200_signal"],
        latest["atr_proxy"],
        unrealized_pnl,
        position_fraction
    ]

    model = load_model()

    state_tensor = torch.tensor(
        state,
        dtype=torch.float32
    ).unsqueeze(0)

    with torch.no_grad():

        q_values = model(state_tensor).squeeze()
    
        valid_actions = [0, 1, 2, 3, 4]
        
        if position_fraction <= 0:
        
            valid_actions.remove(3)
            valid_actions.remove(4)
        
        elif position_fraction >= 0.99:
        
            valid_actions.remove(1)
            valid_actions.remove(2)
        
        best_action = valid_actions[0]
        best_q = q_values[best_action]
        
        for action in valid_actions:
        
            if q_values[action] > best_q:
        
                best_q = q_values[action]
                best_action = action
        
        valid_q = torch.tensor([q_values[a] for a in valid_actions])
        
        valid_probs = torch.softmax(valid_q,dim=0)
        
        confidence = float(valid_probs[valid_actions.index(best_action)])
        
        print("Q Values:", q_values)
        print("Valid Actions:", valid_actions)
        
        return {
            "ticker": ticker,
            "action": ACTION_MAP[best_action],
            "price": float(latest["price"])
        }

In [9]:
predict_signal(
    "AAPL",
    position_fraction=0.01,
    unrealized_pnl=-0.90
)

[*********************100%***********************]  1 of 1 completed

Model loaded from models/dqn_trader.pt
Q Values: tensor([1.2181, 1.3113, 1.4092, 1.2096, 0.4449])
Valid Actions: [0, 1, 2, 3, 4]


{'ticker': 'AAPL', 'action': 'BUY 100%', 'price': 272.5735778808594}

In [5]:
predict_signal(
    "AAPL",
    position_fraction=0.99,
    unrealized_pnl=2.00
)

[*********************100%***********************]  1 of 1 completed

Model loaded from models/dqn_trader.pt
Q Values: tensor([0.2679, 0.2672, 0.2649, 0.2704, 0.2492])
Valid Actions: [0, 3, 4]


{'ticker': 'AAPL', 'action': 'SELL_50', 'price': 272.5735778808594}

In [17]:
for ticker in [
    "AAPL",
    "MSFT",
    "GOOG",
    "AMZN",
    "META",
    "INTC",
    "PYPL",
    "BABA",
    "DIS",
    "IBM"
]:

    print(
        ticker,
        predict_signal(
            ticker,
            position_fraction=0.5,
            unrealized_pnl=0.0
        )["action"]
    )

[*********************100%***********************]  1 of 1 completed


Model loaded from models/dqn_trader.pt
Q Values: tensor([0.2411, 0.2414, 0.2390, 0.2424, 0.2390])
Valid Actions: [0, 1, 2, 3, 4]
AAPL SELL_50


[*********************100%***********************]  1 of 1 completed


Model loaded from models/dqn_trader.pt
Q Values: tensor([0.2042, 0.2047, 0.2028, 0.2081, 0.2067])
Valid Actions: [0, 1, 2, 3, 4]
MSFT SELL_50


[*********************100%***********************]  1 of 1 completed


...

Model loaded from models/dqn_trader.pt
Q Values: tensor([0.2816, 0.2818, 0.2787, 0.2787, 0.2790])
Valid Actions: [0, 1, 2, 3, 4]
BABA BUY_50


[*********************100%***********************]  1 of 1 completed


Model loaded from models/dqn_trader.pt
Q Values: tensor([0.2042, 0.2047, 0.2028, 0.2081, 0.2067])
Valid Actions: [0, 1, 2, 3, 4]
DIS SELL_50


[*********************100%***********************]  1 of 1 completed

Model loaded from models/dqn_trader.pt
Q Values: tensor([0.2160, 0.2163, 0.2143, 0.2196, 0.2150])
Valid Actions: [0, 1, 2, 3, 4]
IBM SELL_50
